# Visualization of Interacting Argon Atoms
> Harrison B. Prosper<br>
> March 2026

## Introduction
In this notebook we visualize the motion of argon atoms [1] in a microscopic spherical container. See notebook `argon_simulator.ipynb` for details.


## Coordinate system
For this project, please choose the coordinate system so that $+z$ points upwards and the $x-y$ plane is horizontal. 

## References
 1. A. Rahman, *Correlations in the Motion of Atoms in Liquid Argon*, Phys. Rev. 136, A405, 1964.

## Tips

  * Use __esc r__ to disable a cell
  * Use __esc y__ to reactivate it
  * Use __esc m__ to go to markdown mode
  * Shift + return to execute a cell

In [1]:
import vpython as vp
import numpy as np
import h5py

from comphyslab.graphics import Bag, Scene, Sim, Controls, Zoom,\
CoordinateSystem, Histogram, Graph, ORIGIN, J, K

from comphyslab.newton import propagate_order3, propagate_order4, \
initialize_fcc, min_separation, TLennardJones, \
maxwell_distribution, radial_distribution, radial_density_profile, KB

from comphyslab.vectors import magnitude, unit, dot, line_sphere_intersect

/Users/harry/miniconda3/envs/comphys/lib/python3.13/site-packages/vpython/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>

## Argon Parameters
We'll use the parameters for argon from Ref.[3] (but in SI units):
  * $\sigma = 3.4 \times 10^{-10} \text{ m}$;
  * $\epsilon =  1.657 \times 10^{-21} \text{ J}$;
  * $m = 6.69\times 10^{-26} \text{ kg}$, and
  * $\rho_0 = 1.642 \text{ kg/m}^3$ @ 293 K and one atmosphere. 

In [2]:
# A Rahman Parameters (Phys. Rev. 136, 1964)
liquid = Bag()
liquid.T = 94.4 # K
liquid.rho = 1.374e3  # kg/m^3

gas = Bag()
gas.T = 293.0 # K
gas.rho = 1.642       # kg/m^3
# --------------------------------------
def read_sim_data(filename, rho=liquid.rho, T=liquid.T):

    bg = Bag()
    bg.sigma   = 3.4e-10          # Distance at which potential is zero (m)
    bg.epsilon = 1.657e-21        # Characteristic energy (J)
    bg.mass    = 6.69e-26         # Mass of argon atom (kg)
    bg.equisep = 2**(1/6)         # Equilibrium separation (sigma)
    bg.v0 = float(np.sqrt(bg.epsilon/bg.mass)) # Characteristic speed (m/s)
    bg.t0 = float(bg.sigma/bg.v0)             # Characteristic timescale (s)
    bg.T2K     = bg.mass * bg.v0**2 / KB
    
    with h5py.File(filename, "r") as f:
        bg.dt  = f.attrs["dt"]
        bg.rho = f.attrs["rho"]
        bg.N   = f.attrs["N"]
        bg.R   = f.attrs["R"]
        bg.r   = f["r"][:]             # (I,N,3)
        bg.v   = f["v"][:]             # (I,N,3)

    print(f'density:         {bg.rho*bg.mass/bg.sigma**3:10.3e} kg/m^3')
    print(f'number density:  {bg.rho:10.3e}/sigma^3')
    print(f'number of atoms: {bg.N:5d} atoms, R = {bg.R:6.3f} sigma')

    r_min = min_separation(bg.r[0])
    print(f'min(separation): {r_min:6.3f} sigma '\
          f'(cf. {bg.equisep:6.3f} sigma)\n')

    # Check that we get the requested temperature
    msv = np.mean((bg.v[0]**2).sum(axis=-1))
    bg.vrms = float(np.sqrt(msv))
    
    T_reduced = msv / 3
    Vrms = bg.v0 * bg.vrms
    
    T = T_reduced * bg.T2K
    bg.T = T
    print(f'Vrms:  {bg.v0*bg.vrms:5.1f} m/s,\tT: {bg.T:8.2f} K')
    print(f'Vrms:  {bg.vrms:5.1f} v0,\tT: {T_reduced:8.2f}')
 
    return bg

##  Define the initial state of the system

In [3]:
def define_initial_state(bg):
    # --------------------------------------------
    # Simulation Constants
    # --------------------------------------------
    bg.size  = bg.R       # Scale of scene (units of sigma)
   
    # For histogram and graphs
    bg.draw_rate = 5   
    bg.bufsize= 200
    bg.bufsize_rho= 1000
    
    # For velocity distribution
    bg.vbins = 40
    bg.vmin  = 0.0
    bg.vmax  = 8.0
    bg.vymin = 0.0
    bg.vymax = 0.20

    # For radial distribution
    bg.rmin  = 0.0
    bg.rmax  = float(np.ceil(bg.R))
    bg.rbins = int(10 * bg.rmax)
    bg.rymin = 0.0
    bg.rymax = 3.0

    # For radial density profile
    bg.rbins_rho = int(5 * bg.rmax)
    bg.rymin_rho = 0.0
    bg.rymax_rho = 0.2
    
    # Container parameters
    bg.Rcore = bg.R * (0.5)**(1/3)
    bg.origin= np.array([0.0, 0.0, 0.0])
    
    # --------------------------------------------
    # Particles
    # --------------------------------------------    
    # Initial state of particles
    
    bg.atom_radius = 0.03 * bg.R 
    bg.mass  = np.ones(bg.N)      # Masses of particles (units of mass)
    bg.charge= np.ones(bg.N)      # Masses of atoms (units of mass)
    # --------------------------------------------
    # Force
    # --------------------------------------------
    bg.k = 1.0
    bg.law = TLennardJones
    
    return bg

### Define the scene

In [4]:
def build_scene(bg):
    bg.rate   = 30         # No faster than "rate" frames/second
    bg.frame  = 0          # Frame counter
    bg.active = True       # Set event loop active
    bg.update = False      # Controled by Start/Pause button

    # Must cache all scene widgets in bag.gfx
    gfx = bg.gfx # give bag.gfx a shorter name

    # Create empty scene in which to position widgets
    bg.height = 200
    gfx.scene = Scene(
        'Argon Atoms\n', bg.size, up=K, height=int(1.2*bg.height))
    gfx.scene.forward = vp.vector(-1,0,0)
    
    # ----------------------------------------------------
    # Add scene elements
    # ----------------------------------------------------
    gfx.container = vp.sphere(
        pos=vp.vector(0.0,0.0,0.0), 
        radius=bg.R, color=vp.color.cyan, opacity=0.05)

    gfx.container_core = vp.sphere(
        pos=vp.vector(0.0,0.0,0.0), 
        radius=bg.Rcore, color=vp.color.cyan, opacity=0.05)

    # Model each atom as a small sphere
    gfx.atoms = []
    r = bg.r[0]
    v = bg.v[0]
    for i in range(bg.N):
        gfx.atoms.append(vp.sphere(
            pos=vp.vector(*r[i]), 
            color=vp.color.yellow,
            radius=bg.atom_radius))

    # Draw control buttons (Stop, Start/Pause)
    controls = Controls(bag)

    gfx.b_stop  = vp.button(
        text="Stop",
        background=vp.color.red,
        pos=gfx.scene.title_anchor,
        bind=controls.stop)

    gfx.b_start_pause = vp.button(
        text="Start",
        background=vp.color.green,
        pos=gfx.scene.title_anchor,
        bind=controls.start_pause)

    gfx.zoom = Zoom(gfx.scene)
    
    # Speeds
    mass = bg.mass
    v0   = bg.v0
    
    v_mag = magnitude(v)
    T = np.mean(v_mag**2) / 3  # dimensionless temperature
    y, _ = maxwell_distribution(T, bg.vmin, bg.vmax, bg.vbins)
    bg.T = bg.T2K * T
    
    gfx.hspeed = Histogram(
        title=f'\nSpeed (T: {bg.T:8.2f} K)', 
        xtitle=f'Speed (v0)', 
        ytitle='f(v)',
        nbins=bg.vbins, 
        xmin=bg.vmin, 
        xmax=bg.vmax, 
        ymin=bg.vymin,
        ymax=bg.vymax,
        bufsize=bg.bufsize,
        height=bg.height,
        width=1.2*bg.height,
        align='left',
        density=True)

    bg.gfx.hspeed.fill(v_mag)
    bg.gfx.hspeed.draw()

    # Maxwell distribution
    bg.gfx.theory = Graph(
        title='Speed distribution', 
        xtitle='Speed (v0 units)', 
        ytitle='Density', 
        npoints=bg.vbins, 
        xmin=bg.vmin, 
        xmax=bg.vmax, 
        graph=gfx.hspeed.g)

    bg.gfx.theory.fill(y)
    bg.gfx.theory.draw() 

    # Radial distribution
    gfx.gradial = Graph(
        title='Radial distribution', 
        xtitle=f'r (units: {1e9*bg.sigma:6.2f} nm)', 
        ytitle='g(r)', 
        npoints=bg.rbins, 
        xmin=bg.rmin, 
        xmax=bg.rmax,
        ymin=bg.rymin, 
        ymax=bg.rymax,
        color=vp.color.blue,
        height=bg.height,
        width=1.2*bg.height,
        align='left',
        bufsize=bg.bufsize)

    y, _ = radial_distribution(
        bg.rho, r, rmax=bg.rmax, nbins=bg.rbins, rcore=bg.Rcore)
    gfx.gradial.fill(y)
    gfx.gradial.draw()


    # Radial density profile
    gfx.gradialrho = Graph(
        title='Radial density', 
        xtitle=f'r (units: {1e9*bg.sigma:6.2f} nm)', 
        ytitle='rho(r)', 
        npoints=bg.rbins_rho, 
        xmin=bg.rmin, 
        xmax=bg.rmax,
        ymin=bg.rymin_rho, 
        #ymax=bg.rymax_rho,
        color=vp.color.blue,
        height=bg.height,
        width=1.2*bg.height,
        align='left',
        bufsize=bg.bufsize_rho)

    y, _ = radial_density_profile(r, bg.R, nbins=bg.rbins_rho)
    gfx.gradialrho.fill(y)
    gfx.gradialrho.draw()

## Scene update function

The $\texttt{update\_scene}$ function is responsible for updating all widgets in a scene and the $\texttt{rate}(n)$ function controls the frequency of updates. Updates will occur no faster than $n$ / second.

The standard update loop has the form:

```python
    while active:
        
        if update:
            
            update_scene(bg) 
            
        vp.rate(30) # allow updates no faster than 30 Hz
```

In [5]:
def update(bg):

    gfx =  bg.gfx
    frame = bg.frame

    # Get data for current frame
    try:
        r = bg.r[frame]
        v = bg.v[frame]
    except:
        bg.update = False
        return

    # Loop over graphical objects and update their
    # positions.
    
    for i, atom in enumerate(gfx.atoms):
        x, y, z = r[i]
        atom.pos.x = x
        atom.pos.y = y
        atom.pos.z = z

    # Update histogram and graphs
    if frame % bg.draw_rate == 0:

        v_mag = magnitude(v)
        gfx.hspeed.fill(v_mag)

        y, _ = radial_distribution(
            bg.rho, r, rmax=bg.rmax, nbins=bg.rbins, rcore=bg.Rcore)
        gfx.gradial.fill(y)

        T = np.mean(v_mag**2) / 3  # dimensionless temperature
        y, _ = maxwell_distribution(T, bg.vmin, bg.vmax, bg.vbins)
        gfx.theory.fill(y)

        y, _ = radial_density_profile(r, bg.R, nbins=bg.rbins_rho)
        gfx.gradialrho.fill(y)

        gfx.hspeed.draw()
        gfx.gradial.draw()
        gfx.theory.draw()
        gfx.gradialrho.draw()

## RUN

In [9]:
bag = read_sim_data(filename='argon_solid.h5')

define_initial_state(bag)

build_scene(bag)

sim = Sim(bag, update)

sim.run()

density:          1.443e+03 kg/m^3
number density:   8.476e-01/sigma^3
number of atoms:   297 atoms, R =  4.373 sigma
min(separation):  1.057 sigma (cf.  1.122 sigma)

Vrms:  222.5 m/s,	T:    79.93 K
Vrms:    1.4 v0,	T:     0.67


<IPython.core.display.Javascript object>

Animation ended!
